# F4 — Auditoría reproducible de MetroPT-3
**Grupo 10 · UNAB**

Ejecutar de arriba abajo con el CSV original. No se fabrican resultados. El notebook produce evidencias pequeñas para adjuntar a GitHub.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, json, platform, timeit
from datetime import datetime, timezone
from IPython.display import display
RUTA = Path('MetroPT3(AirCompressor).csv')  # Cambiar solo si el CSV tiene otro nombre o ubicación
assert RUTA.exists(), f'No se encontró el CSV: {RUTA.resolve()}'
OUT=Path('evidencias_metropt3'); OUT.mkdir(exist_ok=True)
print('Archivo:', RUTA.resolve(), '| Tamaño MB:', round(RUTA.stat().st_size/1e6,2))

## 1. Carga y estructura

In [ ]:
df=pd.read_csv(RUTA, low_memory=False)
df.columns=df.columns.str.strip()
print('Filas:',len(df),'Columnas:',len(df.columns)); display(df.head()); display(df.dtypes.to_frame('tipo'))
assert 'timestamp' in df.columns, 'No existe columna timestamp; revisar separador CSV'


## 2. Auditoría: nulos, duplicados y fechas

In [ ]:
nulos=df.isna().sum().astype(int)
duplicados=int(df.duplicated().sum())
fechas=pd.to_datetime(df['timestamp'], errors='coerce')
fechas_invalidas=int(fechas.isna().sum())
fechas_duplicadas=int(fechas.dropna().duplicated().sum())
fuera_orden=int((fechas.diff().dropna()<pd.Timedelta(0)).sum())
resumen={'archivo':RUTA.name,'registros':int(len(df)),'columnas':int(len(df.columns)),
 'nulos_por_columna':{k:int(v) for k,v in nulos.items()},'total_celdas_nulas':int(nulos.sum()),
 'filas_con_algun_nulo':int(df.isna().any(axis=1).sum()),'filas_duplicadas_exactas':duplicados,
 'fechas_invalidas':fechas_invalidas,'timestamps_duplicados':fechas_duplicadas,
 'saltos_temporales_hacia_atras':fuera_orden,
 'fecha_minima':str(fechas.min()),'fecha_maxima':str(fechas.max()),
 'versiones':{'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__}}
print(json.dumps(resumen,ensure_ascii=False,indent=2))
(OUT/'auditoria_metropt3.json').write_text(json.dumps(resumen,ensure_ascii=False,indent=2),encoding='utf-8')
nulos.rename('nulos').to_csv(OUT/'nulos_por_columna.csv')

## 3. Resumen de sensores y consistencia

In [ ]:
sensores=[c for c in ['TP2','TP3','H1','DV_pressure','Reservoirs','Oil_temperature','Motor_current'] if c in df]
assert sensores, 'No se detectaron sensores esperados'
numericos=df[sensores].apply(pd.to_numeric,errors='coerce')
estadisticas=numericos.describe().T
estadisticas['mediana']=numericos.median()
display(estadisticas)
estadisticas.to_csv(OUT/'estadisticas_sensores.csv')
# Se reportan extremos estadísticos sin eliminarlos automáticamente
q1=numericos.quantile(.25); q3=numericos.quantile(.75); iqr=q3-q1
extremos=((numericos.lt(q1-1.5*iqr)) | (numericos.gt(q3+1.5*iqr))).sum()
extremos.rename('cantidad_fuera_de_1_5_IQR').to_csv(OUT/'extremos_iqr.csv')
print('Extremos por sensor (criterio exploratorio IQR):');display(extremos)

## 4. Comparación de algoritmos sobre una condición idéntica
Se cuenta cuántas observaciones de `Motor_current` superan su mediana. La versión recursiva divide el arreglo en mitades para evitar desbordamiento de recursión.

In [ ]:
x=numericos['Motor_current'].dropna().to_numpy(dtype=float) if 'Motor_current' in numericos else numericos.iloc[:,0].dropna().to_numpy(dtype=float)
u=float(np.median(x))
def iterativo(a,umbral): return sum(v>umbral for v in a)
def recursivo(a,umbral):
    if len(a)==0:return 0
    if len(a)<=512:return iterativo(a,umbral)
    m=len(a)//2
    return recursivo(a[:m],umbral)+recursivo(a[m:],umbral)
def vectorizado(a,umbral):return int(np.count_nonzero(a>umbral))
filas=[]
for tam in [1000,10000,100000,min(1000000,len(x))]:
    a=x[:tam]; esperado=vectorizado(a,u)
    assert iterativo(a,u)==recursivo(a,u)==esperado
    for nombre,fn in [('iterativo',iterativo),('recursivo',recursivo),('vectorizado',vectorizado)]:
        rep=3 if tam>=100000 else 5
        t=timeit.repeat(lambda:fn(a,u),number=1,repeat=rep)
        filas.append({'n':tam,'algoritmo':nombre,'conteo':esperado,'milisegundos_mediana':float(np.median(t)*1000)})
bench=pd.DataFrame(filas);display(bench)
bench.to_csv(OUT/'benchmark_metropt3.csv',index=False)

## 5. Pruebas de consistencia y exportación

In [ ]:
pruebas={
 'dataset_no_vacio':len(df)>0,
 'timestamp_presente':'timestamp' in df,
 'fechas_parseadas':len(fechas)==len(df),
 'nulos_contabilizados':int(nulos.sum())==int(df.isna().sum().sum()),
 'duplicados_no_negativos':duplicados>=0,
 'sensores_detectados':len(sensores)>0,
 'estadisticas_generadas':not estadisticas.empty,
 'algoritmos_equivalentes':all(g.conteo.nunique()==1 for _,g in bench.groupby('n')),
 'benchmark_generado':len(bench)>0}
for k,v in pruebas.items():print(('APROBADA' if v else 'FALLIDA'),k)
assert all(pruebas.values()), 'Hay pruebas fallidas'
(OUT/'pruebas_auditoria_metropt3.txt').write_text('
'.join(f'{k}: {"APROBADA" if v else "FALLIDA"}' for k,v in pruebas.items()),encoding='utf-8')
print('LISTO. Subir a GitHub el notebook y los archivos de',OUT.resolve())